# 📖 Notebook 4: Explore & Recommendation

Instagram's **Explore page** shows you posts from accounts you don't follow —  
content you'll probably enjoy based on what you've liked, saved, and engaged with.

Users spend over **50% of their time** on Explore, making it one of Instagram's  
most important features for discovery and engagement.

## Learning Objectives

By the end of this notebook, you'll understand:
- How **collaborative filtering** works ("users like you also liked...")
- **Engagement scoring** — ranking posts by predicted interest
- The difference between **candidate generation** and **ranking**
- How to cache recommendations for low-latency serving
- Trade-offs between freshness and pre-computation

## 🛠️ Setup

Start the infrastructure first:

```bash
cd 06-system-designs/instagram
docker compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `instagram_demo`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import redis
import time
import json
from collections import Counter, defaultdict

# ── Connections ───────────────────────────────────────────
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "instagram_demo",
    "user": "demo",
    "password": "demo"
}

REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

def get_db():
    return psycopg2.connect(**DB_CONFIG)

def get_redis():
    return redis.Redis(**REDIS_CONFIG)

# Test connections
try:
    conn = get_db()
    cur = conn.cursor()
    cur.execute("SELECT COUNT(*) FROM user_interactions")
    print(f"✅ PostgreSQL — {cur.fetchone()[0]} user interactions")
    cur.execute("SELECT COUNT(*) FROM likes")
    print(f"   {cur.fetchone()[0]} likes")
    cur.execute("SELECT COUNT(*) FROM posts")
    print(f"   {cur.fetchone()[0]} posts")
    conn.close()
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")

try:
    r = get_redis()
    r.ping()
    print(f"✅ Redis connected")
except Exception as e:
    print(f"❌ Redis failed: {e}")

## 🤔 The Explore Problem

The feed shows posts from people you **already follow**.  
Explore shows posts from people you **don't follow but might like**.

How do we decide what to show? Two main signals:

1. **What you've engaged with** — posts you liked, saved, commented on, or spent time viewing
2. **What similar users like** — if users with similar tastes liked something, you might too

The process has two stages:

```
Stage 1: CANDIDATE GENERATION          Stage 2: RANKING
"Find 1000 posts you MIGHT like"       "Pick the best 50 from those 1000"

┌──────────────┐                       ┌──────────────┐
│ Collaborative│                       │              │
│ Filtering    │──┐                    │  Engagement  │
│              │  │                    │  Scoring     │
├──────────────┤  ├──► 1000 posts ──►  │  Model       │──► Top 50 posts
│ Content      │  │    candidates      │              │
│ Similarity   │──┘                    │              │
│              │                       └──────────────┘
└──────────────┘
```

Let's build each stage.

## 🤝 Stage 1: Collaborative Filtering

**Collaborative filtering** is the simplest recommendation approach:  
"Users who liked the same posts as you also liked these other posts."

The algorithm:
1. Find posts that User A has liked
2. Find other users who liked the same posts ("similar users")
3. Find posts those similar users liked that User A hasn't seen
4. Rank by how many similar users liked each post

```
You liked posts: [10, 25, 42]

User 7 also liked posts [10, 25, 99, 150]      ← similar to you!
User 12 also liked posts [25, 42, 88, 200]      ← similar to you!
User 30 liked posts [300, 400]                   ← NOT similar

Candidates: posts [99, 150, 88, 200] (liked by similar users, not by you)
```

In [ ]:
def find_similar_users(user_id: int, min_overlap: int = 2) -> list:
    """
    Find users who have liked many of the same posts as this user.
    Returns users sorted by overlap count (most similar first).
    """
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    cur.execute("""
        -- Step 1: Get posts that this user liked
        WITH my_likes AS (
            SELECT post_id FROM likes WHERE user_id = %s
        ),
        -- Step 2: Find other users who liked the same posts
        similar_users AS (
            SELECT l.user_id AS similar_user_id,
                   COUNT(*) AS overlap_count
            FROM likes l
            JOIN my_likes ml ON ml.post_id = l.post_id
            WHERE l.user_id != %s
            GROUP BY l.user_id
            HAVING COUNT(*) >= %s
            ORDER BY COUNT(*) DESC
            LIMIT 20
        )
        SELECT s.similar_user_id, s.overlap_count,
               u.username, u.display_name
        FROM similar_users s
        JOIN users u ON u.id = s.similar_user_id
        ORDER BY s.overlap_count DESC
    """, (user_id, user_id, min_overlap))

    results = cur.fetchall()
    conn.close()
    return results

# Find users similar to User 1
print("🔍 Finding users similar to User 1...\n")
similar = find_similar_users(user_id=1, min_overlap=1)

if similar:
    print(f"Found {len(similar)} similar users:")
    print(f"{'User':<20} {'Overlap':>10}")
    print("-" * 32)
    for s in similar[:10]:
        print(f"{s['display_name']:<20} {s['overlap_count']:>10} shared likes")
else:
    print("No similar users found (need more interaction data).")
    print("This is expected with our small demo dataset.")

# init.sql seeds 500 likes across 50 users and 180 posts, so overlap at
# min_overlap=1 is effectively guaranteed. If it is not, collaborative
# filtering below has nothing to work with and every later cell silently
# falls through to the popularity path.
assert similar, (
    "no users share a like with user 1 — the `likes` table did not seed "
    "correctly, and the collaborative-filtering sections below cannot run")
assert all(s["similar_user_id"] != 1 for s in similar), "user 1 is not similar to itself"
overlaps = [s["overlap_count"] for s in similar]
assert overlaps == sorted(overlaps, reverse=True), (
    f"similar users must come back most-similar-first, got {overlaps}")

In [ ]:
def get_collaborative_candidates(user_id: int, limit: int = 50) -> list:
    """
    Collaborative filtering: find posts liked by similar users
    that this user hasn't seen yet and doesn't follow the author.

    Also returns `author_avg_likes` — the ranking function needs the author's
    *average* engagement, not this post's, and computing it here costs one
    extra aggregate instead of a query per candidate.
    """
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    cur.execute("""
        -- Posts liked by similar users that I haven't liked
        -- and that aren't from people I already follow
        WITH my_likes AS (
            SELECT post_id FROM likes WHERE user_id = %s
        ),
        my_follows AS (
            SELECT followee_id FROM follows WHERE follower_id = %s
        ),
        similar_users AS (
            SELECT l.user_id AS similar_user_id, COUNT(*) AS overlap
            FROM likes l
            JOIN my_likes ml ON ml.post_id = l.post_id
            WHERE l.user_id != %s
            GROUP BY l.user_id
            HAVING COUNT(*) >= 1
            ORDER BY COUNT(*) DESC
            LIMIT 20
        ),
        -- Author quality: how well this creator's posts do ON AVERAGE.
        author_quality AS (
            SELECT author_id, AVG(like_count)::float AS avg_likes
            FROM posts
            WHERE media_upload_status = 'complete'
            GROUP BY author_id
        )
        SELECT p.id AS post_id, p.author_id, p.caption, p.like_count,
               p.comment_count, p.created_at,
               aq.avg_likes AS author_avg_likes,
               COUNT(DISTINCT su.similar_user_id) AS recommended_by_count
        FROM posts p
        JOIN likes l ON l.post_id = p.id
        JOIN similar_users su ON su.similar_user_id = l.user_id
        JOIN author_quality aq ON aq.author_id = p.author_id
        WHERE p.id NOT IN (SELECT post_id FROM my_likes)
          AND p.author_id NOT IN (SELECT followee_id FROM my_follows)
          AND p.author_id != %s
          AND p.media_upload_status = 'complete'
        GROUP BY p.id, aq.avg_likes
        ORDER BY COUNT(DISTINCT su.similar_user_id) DESC, p.like_count DESC
        LIMIT %s
    """, (user_id, user_id, user_id, user_id, limit))

    results = cur.fetchall()
    conn.close()
    return results

print("🔍 Finding explore candidates for User 1 (collaborative filtering)...\n")
candidates = get_collaborative_candidates(user_id=1)

if candidates:
    print(f"Found {len(candidates)} candidate posts:")
    print(f"{'Post ID':<10} {'Author':<10} {'Likes':>8} {'Reco By':>10} {'Caption':<30}")
    print("-" * 72)
    for c in candidates[:10]:
        print(f"{c['post_id']:<10} User {c['author_id']:<5} {c['like_count']:>8} {c['recommended_by_count']:>10} {c['caption'][:30]}")
else:
    print("No candidates found — this is normal with our small dataset.")
    print("In production, Instagram has billions of interactions to work with.")

# ── Explore must never recommend what the user already follows or liked ──
conn = get_db()
cur = conn.cursor()
cur.execute("SELECT followee_id FROM follows WHERE follower_id = 1")
followed = {row[0] for row in cur.fetchall()}
cur.execute("SELECT post_id FROM likes WHERE user_id = 1")
already_liked = {row[0] for row in cur.fetchall()}
conn.close()

assert candidates, "collaborative filtering produced no candidates for user 1"
bad_author = [c["post_id"] for c in candidates if c["author_id"] in followed]
assert not bad_author, f"Explore surfaced posts from followed accounts: {bad_author[:5]}"
bad_liked = [c["post_id"] for c in candidates if c["post_id"] in already_liked]
assert not bad_liked, f"Explore surfaced posts user 1 already liked: {bad_liked[:5]}"
assert all(c["author_avg_likes"] is not None for c in candidates), (
    "every candidate needs author_avg_likes, or the ranking function's "
    "author-quality term silently becomes a duplicate of engagement")
print(f"\n✅ {len(candidates)} candidates, none from the {len(followed)} accounts "
      f"user 1 follows and none already liked")

## 📈 Stage 2: Engagement Scoring

Collaborative filtering gives us **candidates** — posts the user might like.  
But we need to **rank** them: which candidate should appear first?

Instagram uses machine learning for this, but we can build a simple scoring model  
that captures the key ideas:

```
relevance(post) = (engagement_rate × 0.4)
                + (recency_score   × 0.3)
                + (author_quality  × 0.2)

final_rank      = greedy pass over relevance, subtracting 0.1 for every post
                  already placed by the same author
```

| Signal | What It Measures | Why It Matters |
|--------|-----------------|----------------|
| **Engagement rate** | Log-scaled like count | High engagement = interesting content |
| **Recency** | How recent the post is | Fresh content is more relevant |
| **Author quality** | Author's *average* engagement | Consistently good creators, not one lucky viral post |
| **Diversity** | Spread across different authors | Avoid showing 10 posts from 1 person |

Note where diversity lives. It is **not** a term in `relevance`, and this is the
single easiest thing to get wrong here. Diversity is a property of the *output
list*, not of a post: whether post #42 is "diverse" depends entirely on what you
already decided to show above it. A per-post function cannot know that — it would
have to award the bonus in whatever order the SQL happened to return rows, before
the final ordering exists, and the subsequent sort would then shuffle the bonuses
into meaningless positions.

So relevance is scored per post, and diversity is applied **while building the
final list**: take the best remaining post, discount everything else by the same
author, repeat. That is a greedy re-rank (the simplest member of the MMR family).

In [ ]:
import math

DIVERSITY_PENALTY = 0.1   # subtracted per post already shown from the same author

def score_post(post: dict) -> float:
    """
    Relevance of ONE post, independent of what else we are showing.
    Higher = better. Diversity deliberately lives in rerank_for_diversity().
    """
    # ── Engagement (0 to 1) ──────────────────────────────────
    # Log scale so a viral post doesn't dominate everything else.
    engagement = min(math.log(max(post["like_count"], 1) + 1) / 12.0, 1.0)

    # ── Recency score (0 to 1) ───────────────────────────────
    # Posts from the last few hours score ~1.0, decaying to 0 over 7 days.
    age_hours = (time.time() - post["created_at"].timestamp()) / 3600
    recency = max(0.0, 1.0 - (age_hours / 168))

    # ── Author quality (0 to 1) ──────────────────────────────
    # The author's AVERAGE like count, supplied by the candidate query. Falling
    # back to this post's own like_count would make this term a duplicate of
    # `engagement` — the same number counted twice at 0.6 total weight, with the
    # docs still claiming 0.4/0.2. Assert rather than silently degrade.
    avg_likes = post.get("author_avg_likes")
    assert avg_likes is not None, (
        f"post #{post.get('post_id')} has no author_avg_likes — the candidate "
        f"query must supply it")
    author_quality = min(math.log(max(float(avg_likes), 1.0) + 1) / 12.0, 1.0)

    # ── Weighted relevance ───────────────────────────────────
    return round(engagement * 0.4 + recency * 0.3 + author_quality * 0.2, 4)


def rerank_for_diversity(scored: list, penalty: float = DIVERSITY_PENALTY) -> list:
    """
    Greedy diversity re-rank over (relevance, post) pairs.

    Pick the best remaining post, record its author, then discount every other
    post by that author before picking again. Because the penalty depends on
    what has already been *placed*, it can only be applied here — not inside
    score_post(), where the output order does not exist yet.
    """
    remaining = list(scored)
    shown_by_author = defaultdict(int)
    ordered = []

    while remaining:
        best_i, best_adj = 0, None
        for i, (relevance, post) in enumerate(remaining):
            adjusted = relevance - penalty * shown_by_author[post["author_id"]]
            if best_adj is None or adjusted > best_adj:
                best_i, best_adj = i, adjusted
        _, post = remaining.pop(best_i)
        shown_by_author[post["author_id"]] += 1
        ordered.append((round(best_adj, 4), post))

    return ordered


def top_author_share(rows: list, n: int = 10) -> float:
    """Fraction of the top-n slots taken by the single most-repeated author."""
    authors = [post["author_id"] for _, post in rows[:n]]
    return max(Counter(authors).values()) / len(authors)


# Score our candidates
if candidates:
    print("📊 Scoring candidates...\n")
    scored = [(score_post(c), c) for c in candidates]
    scored.sort(key=lambda x: x[0], reverse=True)

    ranked = rerank_for_diversity(scored)

    print(f"{'Score':>7} {'Post':>6} {'Author':>8} {'Likes':>8} {'Caption':<35}")
    print("-" * 70)
    for score, post in ranked[:10]:
        print(f"{score:>7.4f} #{post['post_id']:<5} User {post['author_id']:<3} "
              f"{post['like_count']:>8} {post['caption'][:35]}")

    # ── Diversity has to change the output, or it is decoration ──────────
    before, after = top_author_share(scored), top_author_share(ranked)
    print(f"\n   Largest single-author share of the top 10: "
          f"{before:.0%} by relevance alone → {after:.0%} after re-ranking")

    assert after <= before, (
        f"the diversity re-rank increased author concentration "
        f"({before:.0%} → {after:.0%}) — it is making the feed worse")
    assert ranked[0][1]["post_id"] == scored[0][1]["post_id"], (
        "the single most relevant post should still lead the feed — a greedy "
        "re-rank never penalises its first pick")
    assert len(ranked) == len(scored), "re-ranking must not drop or duplicate posts"
    assert {p["post_id"] for _, p in ranked} == {p["post_id"] for _, p in scored}
else:
    print("No candidates to score — the popularity fallback below covers this case.")

## 🔥 Fallback: Popularity-Based Recommendations

Collaborative filtering needs enough user interactions to work.  
For **new users** (the "cold start" problem) or when we don't have enough data,  
we fall back to **popularity-based** recommendations.

This simply shows the most-liked posts from the past 24–48 hours  
that the user hasn't already seen.

In [ ]:
def get_popular_posts(user_id: int, hours: int = 48, limit: int = 30) -> list:
    """
    Popularity-based recommendations (cold start fallback).
    Returns trending posts from the last N hours that the user
    hasn't already seen (not in their feed, not from followed users).
    """
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    cur.execute("""
        SELECT p.id AS post_id, p.author_id, p.caption,
               p.like_count, p.comment_count, p.created_at,
               u.username, u.display_name,
               aq.avg_likes AS author_avg_likes
        FROM posts p
        JOIN users u ON u.id = p.author_id
        -- Same author-quality signal the collaborative path supplies, so the
        -- fallback candidates can go through the identical scoring function.
        JOIN (SELECT author_id, AVG(like_count)::float AS avg_likes
              FROM posts
              WHERE media_upload_status = 'complete'
              GROUP BY author_id) aq ON aq.author_id = p.author_id
        WHERE p.created_at > NOW() - (%s || ' hours')::interval
          AND p.author_id NOT IN (
              SELECT followee_id FROM follows WHERE follower_id = %s
          )
          AND p.author_id != %s
          AND p.media_upload_status = 'complete'
        ORDER BY p.like_count DESC
        LIMIT %s
    """, (hours, user_id, user_id, limit))

    results = cur.fetchall()
    conn.close()
    return results

print("🔥 Popular posts (fallback for cold start):\n")
popular = get_popular_posts(user_id=1, hours=720)  # wider window for demo

if popular:
    print(f"{'Rank':<6} {'Likes':>8} {'Author':<20} {'Caption':<35}")
    print("-" * 72)
    for i, p in enumerate(popular[:10], 1):
        print(f"{i:<6} {p['like_count']:>8} {p['display_name']:<20} {p['caption'][:35]}")
else:
    print("No popular posts found in the time window.")

# ── The cold-start path is a fallback; it still has to obey the same rules ──
assert popular, "the popularity fallback returned nothing — cold-start users get an empty page"
likes = [pop["like_count"] for pop in popular]
assert likes == sorted(likes, reverse=True), f"popular posts must be most-liked-first: {likes[:5]}"
assert all(pop["author_id"] not in followed for pop in popular), (
    "the fallback surfaced accounts user 1 already follows")
assert all(pop["author_avg_likes"] is not None for pop in popular), (
    "fallback candidates must carry author_avg_likes so they can share score_post()")
print(f"\n✅ {len(popular)} fallback posts, all scoreable by the same ranking function")

## 🔢 The Signal Underneath: Keeping `like_count` Honest

Every ranking function above leans on `posts.like_count` — it is 0.4 of the
relevance score directly and another 0.2 through author quality. That column is
a **denormalised counter**: a cached answer to
`SELECT COUNT(*) FROM likes WHERE post_id = ...`. It exists because running that
COUNT on every feed render would be absurd, and it drifts the moment two people
tap the heart at the same time.

The tempting implementation is read-modify-write:

```python
count = SELECT like_count FROM posts WHERE id = 42     # read
count = count + 1                                       # modify (in Python!)
UPDATE posts SET like_count = count WHERE id = 42       # write
```

Two concurrent likers both read `10`, both write `11`, and one like vanishes.
This is a **lost update**. It is completely silent — no error, no exception, no
log line. Just a number that is quietly too small, feeding a ranking model that
now under-rates the post and shows it to fewer people.

The fix is to never let the value leave the database:

```sql
UPDATE posts SET like_count = like_count + 1 WHERE id = 42
```

PostgreSQL takes a row lock for the duration of that statement, so concurrent
increments serialise instead of overwriting each other.

Let's run both versions with real concurrent connections and count the damage.

In [ ]:
import threading
from concurrent.futures import ThreadPoolExecutor

LIKERS = 25   # 25 users tap the heart at the same instant


def make_demo_post(caption: str) -> int:
    conn = get_db()
    cur = conn.cursor()
    cur.execute(
        """INSERT INTO posts (author_id, caption, media_type, media_key, like_count)
           VALUES (1, %s, 'photo', 'photos/user_1/counter_demo.jpg', 0)
           RETURNING id""",
        (caption,)
    )
    post_id = cur.fetchone()[0]
    conn.commit()
    conn.close()
    return post_id


def like_read_modify_write(conn, post_id: int):
    """❌ Reads the counter into Python, adds one, writes the result back."""
    cur = conn.cursor()
    cur.execute("SELECT like_count FROM posts WHERE id = %s", (post_id,))
    current = cur.fetchone()[0]
    time.sleep(0.01)          # the window every real request has between read and write
    cur.execute("UPDATE posts SET like_count = %s WHERE id = %s", (current + 1, post_id))
    conn.commit()


def like_atomic(conn, post_id: int):
    """✅ The increment never leaves the database."""
    cur = conn.cursor()
    cur.execute("UPDATE posts SET like_count = like_count + 1 WHERE id = %s", (post_id,))
    conn.commit()


def run_concurrent(like_fn, post_id: int, n: int = LIKERS) -> int:
    """
    Open all n connections FIRST, then release them together on a barrier.
    Without the barrier, connection setup staggers the threads enough that the
    race may not happen at all — and a race demo that does not race teaches
    the opposite of what it claims.
    """
    barrier = threading.Barrier(n)

    def worker(_):
        conn = get_db()
        try:
            barrier.wait(timeout=30)
            like_fn(conn, post_id)
        finally:
            conn.close()

    with ThreadPoolExecutor(max_workers=n) as pool:
        list(pool.map(worker, range(n)))

    conn = get_db()
    cur = conn.cursor()
    cur.execute("SELECT like_count FROM posts WHERE id = %s", (post_id,))
    final = cur.fetchone()[0]
    conn.close()
    return final


bad_post = make_demo_post("Counter demo — read-modify-write ❌")
bad_count = run_concurrent(like_read_modify_write, bad_post)
print(f"❌ read-modify-write : {LIKERS} likes → like_count = {bad_count:<3} "
      f"({LIKERS - bad_count} lost)")

good_post = make_demo_post("Counter demo — atomic increment ✅")
good_count = run_concurrent(like_atomic, good_post)
print(f"✅ atomic UPDATE     : {LIKERS} likes → like_count = {good_count:<3} "
      f"({LIKERS - good_count} lost)")

# ── The lab only teaches something if the broken version actually breaks ──
assert bad_count < LIKERS, (
    f"read-modify-write lost nothing ({bad_count}/{LIKERS}) — this cell is no "
    f"longer reproducing the race it is about")
assert good_count == LIKERS, (
    f"the atomic increment must be exact, got {good_count}/{LIKERS}")

print(f"\n💡 Ranking reads like_count directly, so a {LIKERS - bad_count}-like "
      f"undercount is a")
print(f"   silently wrong Explore feed — not just a wrong badge on the screen.")
print(f"\n   Two more steps a real system takes, both out of scope here:")
print(f"   • The likes table is the source of truth; a reconciliation job")
print(f"     periodically resets like_count from COUNT(*) to repair drift.")
print(f"   • A celebrity post is a hot row — every increment queues on the same")
print(f"     row lock. Production absorbs the writes in Redis (INCR) or sharded")
print(f"     counter rows and flushes the total to PostgreSQL asynchronously,")
print(f"     trading exactness-right-now for throughput.")

## 💾 Caching Explore Results

Computing recommendations is expensive (lots of SQL joins, scoring, etc).  
We can't do this on every Explore page load.

Instead, we **pre-compute** recommendations and cache them in Redis:

```
Background job (runs every 15-30 minutes):
  For each active user:
    1. Generate candidates (collaborative filtering)
    2. Score and rank them
    3. Store top 200 in Redis: explore:{user_id}

When user opens Explore:
  1. Read from Redis (instant!)
  2. Filter out posts they've already seen
  3. Return top 30
```

In [ ]:
def precompute_explore(user_id: int):
    """
    Pre-compute and cache Explore recommendations for a user.
    In production, this runs as a batch job every 15-30 minutes.
    """
    r = get_redis()
    start = time.time()

    # Try collaborative filtering first
    candidates = get_collaborative_candidates(user_id, limit=100)

    # Fall back to popularity if not enough candidates
    if len(candidates) < 20:
        popular = get_popular_posts(user_id, hours=720, limit=100)
        # Convert to same format
        for p in popular:
            if not any(c["post_id"] == p["post_id"] for c in candidates):
                candidates.append(p)

    # Score by relevance, sort, then apply diversity while building the list
    scored = [(score_post(c), c) for c in candidates]
    scored.sort(key=lambda x: x[0], reverse=True)
    ranked = rerank_for_diversity(scored)

    # Store in Redis sorted set (score = final post-diversity score)
    explore_key = f"explore:{user_id}"
    r.delete(explore_key)
    cached = ranked[:200]  # keep top 200
    if cached:
        pipeline = r.pipeline()
        for score, post in cached:
            pipeline.zadd(explore_key, {str(post["post_id"]): score})
        pipeline.expire(explore_key, 1800)  # TTL: 30 minutes
        pipeline.execute()

    elapsed = (time.time() - start) * 1000
    print(f"✅ Pre-computed Explore for user {user_id}")
    print(f"   {len(candidates)} candidates scored, {len(cached)} cached "
          f"in Redis (TTL: 30min)")
    print(f"   Computation time: {elapsed:.0f}ms")
    return cached

def get_explore_feed(user_id: int, limit: int = 30) -> list:
    """
    Read pre-computed Explore feed from Redis.
    This is what happens when the user taps the Explore tab.
    """
    r = get_redis()
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    start = time.time()

    # Read top N post IDs from Redis (sorted by score, highest first)
    explore_key = f"explore:{user_id}"
    post_ids_with_scores = r.zrevrange(explore_key, 0, limit - 1, withscores=True)

    if not post_ids_with_scores:
        print(f"Cache miss for user {user_id} — would trigger pre-computation")
        conn.close()
        return []

    # Hydrate post data
    post_ids = [int(pid) for pid, _ in post_ids_with_scores]
    scores = {int(pid): score for pid, score in post_ids_with_scores}

    cur.execute("""
        SELECT p.id AS post_id, p.author_id, p.caption, p.like_count,
               p.created_at, u.username, u.display_name
        FROM posts p
        JOIN users u ON u.id = p.author_id
        WHERE p.id = ANY(%s)
    """, (post_ids,))

    posts = cur.fetchall()
    # Add scores and sort by score
    for p in posts:
        p["explore_score"] = scores.get(p["post_id"], 0)
    posts.sort(key=lambda p: p["explore_score"], reverse=True)

    elapsed = (time.time() - start) * 1000
    conn.close()

    print(f"⚡ Explore feed for user {user_id}: {elapsed:.1f}ms")
    return posts

# Pre-compute and then read
print("Step 1: Pre-compute (background job)\n")
precompute_explore(user_id=1)

print(f"\nStep 2: Read Explore feed (user opens app)\n")
explore = get_explore_feed(user_id=1)

if explore:
    print(f"\n{'Score':>7} {'Likes':>7} {'Author':<20} {'Caption':<35}")
    print("-" * 72)
    for p in explore[:10]:
        print(f"{p['explore_score']:>7.4f} {p['like_count']:>7} {p['display_name']:<20} {p['caption'][:35]}")

# ── The cache has to hold what the batch job computed, in the right order ──
assert explore, "Explore feed is empty right after pre-computation"
scores = [p["explore_score"] for p in explore]
assert scores == sorted(scores, reverse=True), (
    f"the Explore feed must come back highest-score-first, got {scores[:5]}")
assert len(explore) <= 30, f"asked for 30, got {len(explore)}"

cached_ids = [int(pid) for pid in get_redis().zrevrange("explore:1", 0, -1)]
served = {post["post_id"] for post in explore}
assert served == set(cached_ids[:len(served)]), (
    "the served page must be the cached top slice — hydration dropped posts")
ttl = get_redis().ttl("explore:1")
assert 0 < ttl <= 1800, (
    f"the pre-computed set must expire so a stale ranking cannot serve forever, "
    f"TTL={ttl}")
print(f"\n✅ {len(cached_ids)} posts cached, top {len(explore)} served in cached "
      f"order, cache expires in {ttl}s")

## 📊 Explore Architecture Summary

```
                                    ┌──────────────────────────────────────┐
                                    │     BATCH JOB (every 15-30 min)     │
                                    │                                      │
                                    │  1. For each active user:            │
┌────────────┐                      │     - Collaborative filtering (SQL)  │
│ PostgreSQL │◄─────────────────────│     - Score & rank candidates        │
│            │  read interactions   │     - Store top 200 in Redis         │
│ likes      │                      │                                      │
│ follows    │                      │  2. Fallback: popular posts          │
│ posts      │                      │     for cold-start users             │
│ user_inter │                      └───────────────┬──────────────────────┘
└────────────┘                                      │
                                                    │ write
                                                    ▼
┌─────────┐   GET /explore     ┌───────────┐   ┌──────────┐
│ Client  │───────────────────►│  Explore  │──►│  Redis   │
│         │◄───────────────────│  Service  │   │          │
│         │   top 30 posts     │ (hydrate) │   │ explore: │
└─────────┘                    └───────────┘   │ {user_id}│
                                               └──────────┘
```

## 🧪 Simulating User Interaction and Re-ranking

One powerful concept: as the user interacts with Explore content,  
we can **update their recommendations in real-time** (not just every 30 minutes).

If User 1 likes a post from User 30 on Explore, we can immediately  
boost other posts from User 30 and similar content.

In [ ]:
def record_explore_interaction(user_id: int, post_id: int, interaction_type: str):
    """
    Record a user's interaction with an Explore post and adjust the cached
    ranking in place, so the next Explore pull reflects it immediately instead
    of waiting 30 minutes for the next batch run.
    """
    conn = get_db()
    cur = conn.cursor()
    r = get_redis()

    # Save interaction to database
    cur.execute(
        """INSERT INTO user_interactions (user_id, post_id, interaction_type)
           VALUES (%s, %s, %s)""",
        (user_id, post_id, interaction_type)
    )

    # Get the author of the interacted post
    cur.execute("SELECT author_id FROM posts WHERE id = %s", (post_id,))
    row = cur.fetchone()
    if not row:
        conn.rollback()
        conn.close()
        return {"author_id": None, "siblings": [], "boost": 0.0}
    author_id = row[0]

    explore_key = f"explore:{user_id}"
    cached_ids = [int(pid) for pid in r.zrange(explore_key, 0, -1)]

    # ONE query for the author of every cached post. The obvious version of this
    # is a SELECT per cached post — 200 round-trips on every single tap.
    siblings = []
    if cached_ids:
        cur.execute(
            """SELECT id FROM posts
               WHERE author_id = %s AND id = ANY(%s) AND id <> %s""",
            (author_id, cached_ids, post_id)
        )
        siblings = [srow[0] for srow in cur.fetchall()]
    conn.commit()
    conn.close()

    boost = {"like": 0.15, "save": 0.20, "comment": 0.10, "view": 0.02}
    boost_amount = boost.get(interaction_type, 0.05)

    pipe = r.pipeline()
    for sibling_id in siblings:
        pipe.zincrby(explore_key, boost_amount, str(sibling_id))
    # The user has already engaged with this one — re-showing it wastes a slot.
    pipe.zrem(explore_key, str(post_id))
    pipe.execute()

    print(f"👆 User {user_id} {interaction_type}d post #{post_id} (by user {author_id})")
    print(f"   Boosted {len(siblings)} OTHER posts from user {author_id} "
          f"by +{boost_amount}")
    print(f"   Dropped post #{post_id} from the cached set (already engaged)")
    return {"author_id": author_id, "siblings": siblings, "boost": boost_amount}


# Simulate: User 1 likes a post from their Explore feed
if explore:
    r = get_redis()
    explore_key = "explore:1"
    first_post = explore[0]

    before = {int(pid): s for pid, s in r.zrange(explore_key, 0, -1, withscores=True)}
    print("Before interaction:")
    print(f"   Post #{first_post['post_id']} score: {before[first_post['post_id']]:.4f}\n")

    outcome = record_explore_interaction(
        user_id=1, post_id=first_post["post_id"], interaction_type="like")

    after = {int(pid): s for pid, s in r.zrange(explore_key, 0, -1, withscores=True)}

    print("\nAfter interaction:")
    for sibling_id in outcome["siblings"][:5]:
        print(f"   Post #{sibling_id}: {before[sibling_id]:.4f} → "
              f"{after[sibling_id]:.4f}")

    # ── "Updated in real-time" is a claim; check every score that moved ──
    assert first_post["post_id"] not in after, (
        "an already-liked post must not stay in the Explore candidate set")
    for sibling_id in outcome["siblings"]:
        expected = before[sibling_id] + outcome["boost"]
        assert abs(after[sibling_id] - expected) < 1e-6, (
            f"post #{sibling_id} should have gained exactly +{outcome['boost']}: "
            f"{before[sibling_id]:.4f} → {after[sibling_id]:.4f}, expected {expected:.4f}")
    untouched = set(before) - set(outcome["siblings"]) - {first_post["post_id"]}
    moved = [pid for pid in untouched if after[pid] != before[pid]]
    assert not moved, f"posts by other authors should not have moved: {moved[:5]}"

    print(f"\n✅ {len(outcome['siblings'])} sibling post(s) boosted, "
          f"{len(untouched)} untouched, 1 removed")
    print("→ Open RedisInsight to see the updated scores in explore:1")
    print("\n⚠️  Note the honest limit: this boost is not persisted anywhere.")
    print("   The next batch run overwrites explore:1 from scratch, so the")
    print("   session-level signal is lost unless the batch job reads")
    print("   user_interactions — which it does, one run behind.")
else:
    print("No explore posts to interact with.")

## 🧠 Key Takeaways

1. **Two-stage pipeline**: candidate generation (find posts) → ranking (score and sort)
2. **Collaborative filtering**: "users like you also liked..." — simple but effective
3. **Engagement scoring**: weight multiple signals (likes, recency, author quality, diversity)
4. **Cold start fallback**: popularity-based recommendations for new users
5. **Pre-compute + cache**: batch job generates recommendations, Redis serves them instantly
6. **Real-time updates**: boost scores based on user interactions within the session
7. **Diversity is a property of the list, not the post** — it can only be applied
   while building the output, never as a term in a per-post score
8. **Your ranking is only as good as its inputs**: `like_count` is a denormalised
   counter, and a read-modify-write increment silently loses likes under
   concurrency, quietly under-ranking the post

### Interview Tips

- Describe the two-stage pipeline (candidate generation → ranking)
- Mention collaborative filtering as the core approach
- Address the **cold start problem** — what do you show new users?
- Talk about pre-computation vs real-time: batch jobs for base scores, real-time boosts for freshness
- Discuss **diversity** — don't show 10 posts from the same author
- At scale, mention that ML models (neural networks) replace the simple scoring function
- If asked about engagement counters: atomic increments, a reconciliation job
  against the source-of-truth table, and Redis absorption for hot rows

### What This Toy Version Does NOT Do

- **No embeddings.** Real candidate generation is nearest-neighbour search over
  learned post/user vectors; `likes` overlap is the 1990s version
- **No negative feedback.** "Not interested", hides and quick scroll-aways are
  strong signals and are ignored here
- **No per-user batch job.** `precompute_explore` runs for one user on demand;
  at 500M users this is a scheduled pipeline over active users only
- **No integrity or safety filtering.** A production Explore page runs every
  candidate through policy classifiers before it is allowed to be ranked

### Lab Complete! 🎉

You've now built the core systems behind Instagram:
1. ✅ **Photo Upload Pipeline** — pre-signed URLs, thumbnails, CDN caching
2. ✅ **News Feed Generation** — fan-out on write, hybrid celebrity approach
3. ✅ **Stories** — Redis TTL for ephemeral content, view tracking
4. ✅ **Explore & Recommendations** — collaborative filtering, engagement scoring